# ALE-auxiliary coupling diagram

This code generated Figure 3 in the paper [Tip growth in a strongly concentrated aggregation model follows local geodesics](https://arxiv.org/abs/2304.04417).

## Conformal maps and the initial conditions

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.linalg import solve
from scipy.optimize import fsolve
from tqdm import trange
#from numba import jit
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from datetime import datetime

In [ ]:
def wstar(zs, betas):
    """
    Computes the w^* from Pearce, 1991.
    We can use this to find the value of c.
    
    zs    is a list of n points on the circle,
          with the last point equal to 1 + 0j;
    betas is a list of n exponents.

    This involves computing an integral numerically
    with a singularity potentially at one end of the range
    (in the centre of the disc).
    Since we chose a tip as w_n with z_n = 1 and beta_n = -1,
    the boundary has a zero instead of a pole.

    We can handle this using scipy.integrate.quad
    with weight='alg'.
    """
    def unweighted_integrand(t):
        prod = 1
        for i,z in enumerate(zs):
            prod *= (1 - z.conjugate()*t)**(-betas[i])
        return (prod - 1)/t
    integral_re = quad(lambda t : unweighted_integrand(t).real, 0, 1)
    integral_im = quad(lambda t : unweighted_integrand(t).imag, 0, 1)
    # print(f"Computed wstar with error value {integral[1]}")
    return np.exp(integral_re[0] + 1j*integral_im[0])

def ftip(preimage,zs,betas,c):
    """
    Computes the image of `preimage` when `preimage` is
    the preimage of a tip. It's the same method as when
    it's a base, but we do something a slightly different
    to deal with the pole for bases.
    """
    def unweighted_integrand(t):
        prod = 1
        for i,z in enumerate(zs):
            prod *= (1 - z.conjugate()*preimage*t)**(-betas[i])
        return (prod - 1)/t
    integral_re = quad(lambda t : unweighted_integrand(t).real, 0, 1)
    integral_im = quad(lambda t : unweighted_integrand(t).imag, 0, 1)
    return c*preimage*np.exp(integral_re[0] + 1j*integral_im[0])

def fbase(preimage_index,zs,betas,c):
    """
    Computes the image of `preimage` when `preimage` is
    the preimage of a base. Deals carefully with the pole
    at that base itself.
    We take the index instead of the value because then
    we can take the singular term out of the product and
    deal with it separately.
    """
    preimage = zs[preimage_index]
    def unweighted_integrand_1(t):
        # For t in (0,1/2]
        prod = 1
        for i in range(len(zs)):
            prod *= (1 - zs[i].conjugate()*preimage*t)**(-betas[i])
        return (prod - 1)/t
    def unweighted_integrand_2(t):
        # For t in [1/2, 1).
        # The other term becomes (1 - t)^{-1/2}.
        prod = 1
        for i in range(len(zs)):
            if i == preimage_index:
                continue
            else:
                prod *= (1 - zs[i].conjugate()*preimage*t)**(-betas[i])
        return prod/t
    integral1_re = quad(lambda t : unweighted_integrand_1(t).real,0,1/2)
    integral1_im = quad(lambda t : unweighted_integrand_1(t).imag,0,1/2)
    integral1 = integral1_re[0] + 1j*integral1_im[0]
    integral2_re = quad(lambda t : unweighted_integrand_2(t).real,1/2,1,weight='alg',wvar=(0,-1/2))
    integral2_im = quad(lambda t : unweighted_integrand_2(t).imag,1/2,1,weight='alg',wvar=(0,-1/2))
    integral2 = integral2_re[0] + 1j*integral2_im[0]
    integral = integral1 + integral2 - np.log(2)
    return c * preimage * np.exp( integral )

In [ ]:
def get_params(ys):
    """
    In the words of Trefethen, this is "easy to do, though not immediate".
    It involves defining a set of linear equations then solving them.
    """
    eys = np.exp(ys)
    # Construct a matrix
    A = np.zeros((len(ys),len(ys)))
    A[0,0] = -(1 + eys[0])
    A[0,1] = eys[0]
    for i in range(1,len(ys)-1):
        A[i,i-1] = 1
        A[i,i]   = -(1 + eys[i])
        A[i,i+1] = eys[i]
    A[len(ys)-1,len(ys)-2] = 1
    A[len(ys)-1,len(ys)-1] = -(1+eys[len(eys)-1])
    b = np.zeros(len(ys))
    b[len(ys)-1] = 2*np.pi*eys[len(eys)-1]
    phis = np.append(solve(A,b),0.0)
    zs = np.exp( 1j * phis )
    return zs

def get_w_hats(zs,betas,c):
    w_hats = np.empty(len(zs),dtype=np.complex128)
    preimage_type = 0 # 0 and 1 for bases, 2 for a tip.
    # The final wn is a tip, so the first two must be bases.
    for i in range(len(zs)):
        if preimage_type == 2:
            # We have a tip
            w_hats[i] = ftip(zs[i],zs,betas,c)
            preimage_type = 0
        else:
            # We have a base
            w_hats[i] = fbase(i,zs,betas,c)
            preimage_type += 1
    return w_hats

def my_arg(z):
    # argument in the range [0,2pi)
    theta = np.angle(z)
    if theta < 0:
        theta += 2*np.pi
    return theta

def objective_fn(ys,betas,ws):
    zs = get_params(ys)
    c = ws[-1]/wstar(zs,betas)
    w_hats = get_w_hats(zs,betas,c)
    equations = np.zeros(len(ys))
    preimage_type = 0 # 0 and 1 for bases, 2 for a tip.
    # The final wn is a tip, so the first two must be bases.
    # If preimage_type is 0 or 2 we have an equation like (3a),
    # and if preimage_type == 1 we have an equation like (3c).
    for i in range(len(ys)-1):
        if preimage_type == 0:
            # w_m is a base, w_{m-1} is a tip
            equations[i] = np.abs(w_hats[i]) - np.abs(ws[i])
            preimage_type += 1
        elif preimage_type == 1:
            # w_m is a base, w_{n-1} is also a base.
            equations[i] = my_arg(w_hats[i]/w_hats[i-1]) - my_arg(ws[i]/ws[i-1])
            preimage_type += 1
        else:
            # w_m is a tip, w_{m-1} is a base
            equations[i] = np.abs(w_hats[i]) - np.abs(ws[i])
            preimage_type = 0
    angle_sum = np.sum( [betas[i]*my_arg(zs[i]) for i in range(len(zs))] )
    remainder = angle_sum % np.pi
    if remainder > 0.5*np.pi:
        remainder = remainder - np.pi
    equations[len(ys)-1] = remainder
    return equations

### Set the parameters for $K_0$

In [ ]:
# Locations given in degrees (increasing order in [0,2pi)).
degree_locations = [45,135,225]
# Lengths of the slits
lengths   = [1, 1.5, 1]

In [ ]:
locations = [a*np.pi/180 for a in degree_locations]

ws = np.empty(3*len(locations),dtype=np.complex128)
ws[0] = np.exp(-1j*locations[0])
for i in range(1,len(locations)):
    ws[3*i - 2] = np.exp(-1j*locations[i])
    ws[3*i - 1] = np.exp(-1j*locations[i]) / (1 + lengths[i])
    ws[3*i]     = np.exp(-1j*locations[i])
ws[3*len(locations)-2] = np.exp(-1j*locations[0])
ws[3*len(locations)-1] = np.exp(-1j*locations[0]) / (1 + lengths[0])

betas = np.empty(3*len(locations))
preimage_type = 0 # 0 and 1 are bases, 2 is a tip.
for i in range(3*len(locations)):
    if preimage_type == 2:
        betas[i] = -1
        preimage_type = 0
    else:
        betas[i] = 1/2
        preimage_type += 1

# Get the parameters.
ys = fsolve(objective_fn,np.zeros(len(ws)-1),args=(betas,ws))
zs = get_params(ys)
c = ws[-1] / wstar(zs,betas)
print(objective_fn(ys,betas,ws))

In [ ]:
# fig, ax = plt.subplots()
# unit_circle = Circle((0,0),1,fill=False)
# ax.add_patch(unit_circle)
# # for z in zs:
# #     plt.scatter(z.real,z.imag,marker='x',color='red')
# ## Draw the envelope
# sigma = 0.00001
# envelope = np.exp( 1j * np.linspace(0,2*np.pi,1000) - sigma )
# for i in range(len(envelope)):
#     envelope[i] = 1/ftip(envelope[i],zs,betas,c)
#     plt.scatter(envelope[i].real, envelope[i].imag, marker='.', color='blue',s=2)
# plt.gca().set_aspect('equal')
# plt.show()

In [ ]:
# All of ws, zs, betas and c are global variables.
def phi0(z):
    # Takes z in the exterior disc
    # to its image under phi0.
    return 1/ftip(1/z,zs,betas,c)

#@jit(nopython=True)
def phi0_prime(z,image):
    # Does it make more sense when eta > 0 to compute the
    # reciprocal of the derivative instead?
    derivative = image/z
    for m in range(len(zs)):
        derivative *= (z - zs[m].conjugate())**(-betas[m])
    return derivative

#@jit(nopython=True)
def phi0_prime_reciprocal(z,image):
    output = z/image
    for m in range(len(zs)):
        output *= (z - zs[m].conjugate())**(betas[m])
    return output

#@jit(nopython=True)
def slitmap(z,theta,capacity):
    ec = np.exp(capacity)
    w = z * np.exp(-1j*theta)
    return np.exp(1j*theta) * ((0.5*ec/w)*(w+1)**2 * (1 + np.sqrt((w*w + 2*(1 - 2/ec)*w + 1)/((w+1)**2))) - 1)

#@jit(nopython=True)
def slitmap_derivative(z,theta,capacity,fz):
    cosbetac = np.cos(np.asin(2*np.exp(-capacity)*np.sqrt(np.exp(capacity)-1)))
    w = z*np.exp(-1j*theta)
    return (fz/z) * (w-1) / np.sqrt(w*w - 2*cosbetac*w + 1)

#@jit(nopython=True)
def slitmap_derivative_reciprocal(z,theta,capacity,fz):
    cosbetac = np.cos(np.asin(2*np.exp(-capacity)*np.sqrt(np.exp(capacity)-1)))
    w = z*np.exp(-1j*theta)
    return (z/fz) * np.sqrt(w*w - 2*cosbetac*w + 1) / (w-1)

## ALE and auxiliary comparison

In [ ]:
capacity = 0.1
ec = np.exp(capacity)
d = 2*(ec - 1) + 2*ec*np.sqrt(1 - 1/ec)

angle_seq = [1.8, 0.0, 3.7, 1.6, 0.1]

# delta_1 = 0.1
# delta_2 = 0.07
# delta_3 = 0.23
# delta_4 = -0.2
# delta_5 = 0.16

circle_pts = 1000
n = len(angle_seq)

from colorspace import hcl_palettes
my_palette = hcl_palettes().get_palette(name="Batlow")
colours = my_palette.colors(n)

slitdensity = 100 # points plotted per slit.
slit = np.linspace(1+d,1,num=slitdensity,endpoint=False)

cluster = np.empty(shape=slitdensity*n,dtype=np.complex128)
for t in range(n):
    cluster[:t*slitdensity] = slitmap(cluster[:t*slitdensity],angle_seq[n-1-t],capacity)
    cluster[t*slitdensity:(t+1)*slitdensity] = np.exp(1j*angle_seq[n-1-t])*slit
for i in trange(len(cluster)):
    cluster[i] = phi0(cluster[i])

fig, ax = plt.subplots()
ax.set_aspect(1.0)
unit_circle = Circle((0,0),1,fill=False)
ax.add_patch(unit_circle)
for t in range(n):
    ax.scatter(cluster.real[t*slitdensity:(t+1)*slitdensity],cluster.imag[t*slitdensity:(t+1)*slitdensity],
               color=colours[t],s=1)
for k in range(len(locations)):
    arm = np.exp(1j*locations[k]) * np.linspace(1+lengths[k],1,num=slitdensity,endpoint=False)
    ax.scatter(arm.real,arm.imag,color='k',s=0.2)

plt.xlim(-4.5,3.0)
plt.ylim(-3.5,5)

now = datetime.now().isoformat()
fig.savefig(f"{now}.pdf")
plt.show()

In [ ]:
# New rotation for the cluster (in degrees)
rotation = (0.1 + 0.07 + 0.23 - 0.2 + 0.16)*180/np.pi

Re-compute the rotated \Phi_0 map

In [ ]:
# Locations given in degrees (increasing order in [0,2pi)).
degree_locations = [45+rotation,135+rotation,225+rotation]
# Lengths of the slits
lengths   = [1, 1.5, 1]

locations = [a*np.pi/180 for a in degree_locations]

ws = np.empty(3*len(locations),dtype=np.complex128)
ws[0] = np.exp(-1j*locations[0])
for i in range(1,len(locations)):
    ws[3*i - 2] = np.exp(-1j*locations[i])
    ws[3*i - 1] = np.exp(-1j*locations[i]) / (1 + lengths[i])
    ws[3*i]     = np.exp(-1j*locations[i])
ws[3*len(locations)-2] = np.exp(-1j*locations[0])
ws[3*len(locations)-1] = np.exp(-1j*locations[0]) / (1 + lengths[0])

betas = np.empty(3*len(locations))
preimage_type = 0 # 0 and 1 are bases, 2 is a tip.
for i in range(3*len(locations)):
    if preimage_type == 2:
        betas[i] = -1
        preimage_type = 0
    else:
        betas[i] = 1/2
        preimage_type += 1

# Get the parameters.
ys = fsolve(objective_fn,np.zeros(len(ws)-1),args=(betas,ws))
zs = get_params(ys)
c = ws[-1] / wstar(zs,betas)
print(objective_fn(ys,betas,ws))

In [ ]:
capacity = 0.1
ec = np.exp(capacity)
d = 2*(ec - 1) + 2*ec*np.sqrt(1 - 1/ec)

angle_seq = [1.7, -0.07, 3.47, 1.7, -0.14] # Calculated manually.

circle_pts = 1000
n = len(angle_seq)
from colorspace import hcl_palettes
my_palette = hcl_palettes().get_palette(name="Batlow")
colours = my_palette.colors(n)
slitdensity = 100 # points plotted per slit.
slit = np.linspace(1+d,1,num=slitdensity,endpoint=False)
cluster = np.empty(shape=slitdensity*n,dtype=np.complex128)
for t in range(n):
    cluster[:t*slitdensity] = slitmap(cluster[:t*slitdensity],angle_seq[n-1-t],capacity)
    cluster[t*slitdensity:(t+1)*slitdensity] = np.exp(1j*angle_seq[n-1-t])*slit
for i in trange(len(cluster)):
    cluster[i] = phi0(cluster[i])
fig, ax = plt.subplots()
ax.set_aspect(1.0)
unit_circle = Circle((0,0),1,fill=False)
ax.add_patch(unit_circle)
for t in range(n):
    ax.scatter(cluster.real[t*slitdensity:(t+1)*slitdensity],cluster.imag[t*slitdensity:(t+1)*slitdensity],
               color=colours[t],s=1)
for k in range(len(locations)):
    arm = np.exp(1j*locations[k]) * np.linspace(1+lengths[k],1,num=slitdensity,endpoint=False)
    ax.scatter(arm.real,arm.imag,color='k',s=0.2)
plt.xlim(-4.5,2.5)
plt.ylim(-3.5,5)
now = datetime.now().isoformat()
fig.savefig(f"{now}.pdf")
plt.show()